# Anchor Refinery
Created by Quillan Shimp with the use of Claude. Images from Legacy Survey.  

### How to Use
Ensure cutout_vetter.py is in this directory.  
Select SGAML kernel.  
Start the vetting program by running all cells.  
Input your username.  
Use the morphology dropdown to change galaxy type, use the style buttons to change image type.  
Select any number of galaxies. Then assign an alternate morphology, bad anchor attribute, and/or write notes.  
Click the galaxy again while the border is blue to deselect.  
Click affirm all to assign galaxies on page as the given type. 
You may change the number of galaxies per page and number of columns in the Settings cell.  
You may click save to save your progress and exit to end your session.  
### Advice
When initiated, the galaxies will not display correctly. Press any button that changes the display and it will function normally.  
Click save intermittently while vetting (I do so every hundred galaxies), you never know what will happen.  
Write notes before assigning morphology. Once the type is assigned, the galaxy is uninteractable.  
If memory overloads during a session and the page refreshes, don't panic. The tool will reload where you left off and you can save from there.  
Avoid overusing the bad anchor option. Removing the anchor won't remove the galaxy from the classifier! Whether some galaxies should not be classified should be discussed.
If you think of any improvement, let me know!
### The Sample
The first 450 galaxies or so of each type are from Julia's original anchor set. They were vetted with all image settings available. The remaining galaxies used John's SGA-2025 group-centered cutouts (SSL mode). These galaxies have only been vetted once. The final 400 or so irregulars were notably vetted within a 5 hour timespan and likely need careful attention to remove spirals. I avoided classifying grouped galaxies whenever ambiguous, but some galaxy groups survived with potentially unexpected primary galaxies leading to laughably wrong classification.

### 1.
Julia's anchors available in SGA 2025 are generated using Select_Galaxies.ipynb

In [1]:
import numpy as np
import pandas as pd

import glob
import h5py
import os

import matplotlib.pyplot as plt
import matplotlib.image as mpimg # displaying images

from IPython.display import display, clear_output # for display functions
import ipywidgets as widgets # for buttons

from cutout_vetter import CutoutVetter, ReviewerLogin, confusion_matrix_report # interactive image grid

In [2]:
# Taken from /global/homes/q/qshimp/SGA/doc/tutorials/SGA-ssl.ipynb
def build_cutout_index(ssl_dir):
    """
    Return {(region, sgaid): (hdf5_path, row_index)}
    for fast image retrieval.
    """
    files = sorted(glob.glob(os.path.join(ssl_dir, "ssl-cutouts-dr11-*.hdf5")))
    if not files:
        raise FileNotFoundError(f"No cutout files found in {ssl_dir}")

    index = {}
    for f in files:
        filename = os.path.basename(f)
        
        if "dr11-south" in filename:
            region = "dr11-south"
        elif "dr11-north" in filename:
            region = "dr11-north"
        else:
            raise ValueError(f"Could not determine region from filename: {filename}")
            
        with h5py.File(f, "r") as H:
            sgaids = H["sgaid"][:]
            
            for i, sgaid in enumerate(sgaids):
                key = (region, int(sgaid))
                if key in index:
                    print(f"WARNING: duplicate key found: {key}")
                index[key] = (f, i)

    print(f"Total unique (region, SGAID) pairs indexed: {len(index):,}")

    return index

def build_jpg_index(base):
    jpg_index = {}

    for kind in ["Model", "Residual", "Image"]:
        directory = os.path.join(base, kind)

        for path in glob.glob(os.path.join(directory, "*.jpg")):
            filename = os.path.basename(path)
            sgaid = filename.split("_", 1)[0]

            jpg_index.setdefault(int(sgaid), {})[kind] = path

    return jpg_index

JPG_INDEX = build_jpg_index("/pscratch/sd/q/qshimp/Cutouts/sga2025/Anchor_jpgs")
    
# Load legacy survey jpgs
def load_jpg_band(row, band):
    """Load exactly one band ('Model', 'Residual', or 'Image') — avoids
    reading and decoding the other two when they're not being displayed."""
    sgaid = int(row["SGAID"])
    path = JPG_INDEX.get(sgaid, {}).get(band)
    if path is None:
        return None
    return np.flipud(mpimg.imread(path))


def load_jpg(row):
    """Kept for backward compatibility with anything still expecting the 3-tuple."""
    return (
        load_jpg_band(row, "Model"),
        load_jpg_band(row, "Residual"),
        load_jpg_band(row, "Image"),
    )

def lookup_by_morph(morph_label, df):
    return df[df["Morphology"] == morph_label].copy()

# Taken from SGA.qa
def sdss_rgb(imgs, bands, scales=None, m=0.03, Q=20, mnmx=None, clip=True):
    """Convert a list of band images to an RGB array using an arcsinh stretch.

    Default scaling matches the Legacy Survey viewer::

        g -> Blue  (plane 2, scale 6.0)
        r -> Green (plane 1, scale 3.4)
        i -> Red   (plane 0, scale 3.0)
        z -> Red   (plane 0, scale 2.2)

    For ``['g', 'r', 'i', 'z']`` a band-mixing scheme is used so that
    all four bands contribute to all three color channels. For any
    other combination (e.g. ``['g', 'r', 'z']``) each band maps to a
    single plane.

    Parameters
    ----------
    imgs : :class:`list` of :class:`numpy.ndarray`
        Per-band 2-D image arrays, in the same order as `bands`.
    bands : :class:`list` of :class:`str`
        Band names, from ``{'g', 'r', 'i', 'z'}``.
    scales : :class:`dict`, optional
        Override per-band ``(plane, scale)`` tuples.
    m : :class:`float`
        Additive offset applied before the arcsinh stretch.
    Q : :class:`float` or None
        Arcsinh softening parameter. Set to None to skip the stretch
        entirely (linear scaling only).
    mnmx : :class:`tuple`, optional
        Manual ``(min, max)`` linear clip range, used instead of the
        arcsinh/percentile-style normalization.
    clip : :class:`bool`
        If True, clip the output to ``[0, 1]``.

    Returns
    -------
    :class:`numpy.ndarray`
        ``(H, W, 3)`` float32 RGB array.

    """
    _scales = dict(g=(2, 6.0), r=(1, 3.4), i=(0, 3.0), z=(0, 2.2))
    if scales is not None:
        _scales.update(scales)

    I = 0
    for img, band in zip(imgs, bands):
        _, scale = _scales[band]
        I = I + np.maximum(0, img * scale + m)
    I /= len(bands)
    if Q is not None:
        fI = np.arcsinh(Q * I) / np.sqrt(Q)
        I += (I == 0.) * 1e-6
        I = fI / I

    H, W = I.shape
    rgb = np.zeros((H, W, 3), np.float32)

    if list(bands) == ['g', 'r', 'i', 'z']:
        rgbvec = dict(
            g=(0.,   0.,  0.75),
            r=(0.,   0.5, 0.25),
            i=(0.25, 0.5, 0.),
            z=(0.75, 0.,  0.))
        for img, band in zip(imgs, bands):
            _, scale = _scales[band]
            rf, gf, bf = rgbvec[band]
            if mnmx is None:
                v = (img * scale + m) * I
            else:
                v = ((img * scale + m) - mnmx[0]) / (mnmx[1] - mnmx[0])
            if clip:
                v = np.clip(v, 0, 1)
            rgb[:, :, 0] += rf * v
            rgb[:, :, 1] += gf * v
            rgb[:, :, 2] += bf * v
    else:
        for img, band in zip(imgs, bands):
            plane, scale = _scales[band]
            if mnmx is None:
                imgplane = (img * scale + m) * I
            else:
                imgplane = ((img * scale + m) - mnmx[0]) / (mnmx[1] - mnmx[0])
            if clip:
                imgplane = np.clip(imgplane, 0, 1)
            rgb[:, :, plane] = imgplane

    return rgb

In [3]:
SSL_DIR = '/global/cfs/cdirs/desicollab/users/ioannis/SGA/2025/ssl'
CATALOG = '/global/cfs/cdirs/desicollab/users/qshimp/anchors/catalog_4400.csv'
anchor_catalog = pd.read_csv(CATALOG)
cutout_index = build_cutout_index(SSL_DIR)

Total unique (region, SGAID) pairs indexed: 445,693


In [4]:
# Settings
N_COLS = 5
N_PER_PAGE = 20

In [5]:
%matplotlib widget

login = ReviewerLogin(
    morph_options=sorted(anchor_catalog["Morphology"].unique()),
    data_lookup_fn=lookup_by_morph,
    df=anchor_catalog,
    cutout_index=cutout_index,
    sdss_rgb_fn=sdss_rgb,
    load_jpg_fn=load_jpg,
    load_jpg_band_fn=load_jpg_band,
    ncols=N_COLS,
    n_per_page=N_PER_PAGE,
    figsize_per=2,
    n_workers=8
)

In [6]:
matrix = confusion_matrix_report(
    save_dir="/global/cfs/cdirs/desicollab/users/qshimp/anchors",
    morph_options=sorted(anchor_catalog["Morphology"].unique()),
    username="qshimp"
)
matrix

,Elliptical,Irregular,Lenticular,Spiral,Bad anchor
Elliptical,504,13,38,0,2
Irregular,0,0,0,0,0
Lenticular,0,0,0,0,0
Spiral,0,2,9,0,0


# To do:
1. Allow page selection 
2. Create user system
3. Create exit button next to save button that returns to input username screen
5. Create getInfo function to enable users to check progress (matrix style display)
6. Change save from separate csvs to one master file (for each user)

## Other options:
1. Confirmed galaxies disappear instead of red border
2. Quality of life like "Loading..." or "Timed out!" to reduce waiting
3. Unsure button: gives galaxies a tag for a later "review uncertain galaxies" page in late-stage vetting process where majority of galaxies have been reviewed with rigor